In [2]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from transformers import ViTForImageClassification, ViTImageProcessor, get_scheduler
from torch.optim import AdamW
from tqdm import tqdm
import os
import pandas as pd

/home/info-sec-lab/BTP/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## **TRAINING**

**CONFIG**

In [3]:

DATA_DIR = "../snapshots/Train"       # path to your dataset
OUTPUT_DIR = "../checkpoints/vit_only"     # directory to save checkpoints
BATCH_SIZE = 8
EPOCHS = 3
LEARNING_RATE = 5e-5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(OUTPUT_DIR, exist_ok=True)

**PREPROCESSING**

In [4]:
processor = ViTImageProcessor.from_pretrained("facebook/deit-base-patch16-224")

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std)
])

dataset = datasets.ImageFolder(DATA_DIR, transform=train_transforms)
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 31536.12it/s]


**MODEL**

In [5]:
num_labels = len(dataset.classes)
model = ViTForImageClassification.from_pretrained(
    "facebook/deit-base-patch16-224",
    num_labels=num_labels,
    ignore_mismatched_sizes=True
)
model.to(DEVICE)

Some weights of ViTForImageClassification were not initialized from the model checkpoint at facebook/deit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([2]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (intermed

**OPTIMIZER & SCHEDULER**

In [6]:
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
num_training_steps = EPOCHS * len(train_loader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps
)

**TRAINING LOOP**

In [7]:
model.train()
for epoch in range(EPOCHS):
    print(f"\n🔁 Epoch {epoch + 1}/{EPOCHS}")
    total_loss = 0

    for batch in tqdm(train_loader):
        images, labels = batch
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        outputs = model(images, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"✅ Epoch {epoch + 1} Completed | Avg Loss: {avg_loss:.4f}")

    # Save checkpoint
    ckpt_path = os.path.join(OUTPUT_DIR, f"deit_epoch_{epoch + 1}.pt")
    torch.save(model.state_dict(), ckpt_path)
    print(f"💾 Saved checkpoint to {ckpt_path}")

print("\n🎉 Training Complete!")


🔁 Epoch 1/3


100%|██████████| 774/774 [04:06<00:00,  3.13it/s]


✅ Epoch 1 Completed | Avg Loss: 0.6182
💾 Saved checkpoint to ../checkpoints/vit_only/deit_epoch_1.pt

🔁 Epoch 2/3


100%|██████████| 774/774 [03:11<00:00,  4.04it/s]


✅ Epoch 2 Completed | Avg Loss: 0.4927
💾 Saved checkpoint to ../checkpoints/vit_only/deit_epoch_2.pt

🔁 Epoch 3/3


100%|██████████| 774/774 [03:11<00:00,  4.05it/s]

✅ Epoch 3 Completed | Avg Loss: 0.3440
💾 Saved checkpoint to ../checkpoints/vit_only/deit_epoch_3.pt

🎉 Training Complete!


## **TESTING**

**CONFIG**

In [8]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = "../checkpoints/vit_only/deit_epoch_3.pt"   # path to your trained checkpoint
TEST_ROOT = "../snapshots"
BATCH_SIZE = 8
OUTPUT_CSV = "Result_Sheets/vit_only/deit_test_results.csv"

**PREPROCESSING**

In [9]:
processor = ViTImageProcessor.from_pretrained("facebook/deit-base-patch16-224")

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std)
])

Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 33554.43it/s]


**LOAD MODEL**

In [10]:
# use same num_labels as training
NUM_LABELS =  len(os.listdir(os.path.join(TEST_ROOT, "Test_0")))  # only used for init
model = ViTForImageClassification.from_pretrained(
    "facebook/deit-base-patch16-224",
    num_labels=NUM_LABELS,
    ignore_mismatched_sizes=True
)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE)
model.eval()

Some weights of ViTForImageClassification were not initialized from the model checkpoint at facebook/deit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([2]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (intermed

**EVALUATION LOOP** 

In [11]:
results = []

for test_idx in range(10):
    folder = os.path.join(TEST_ROOT, f"Test_{test_idx}")
    if not os.path.exists(folder):
        print(f"⚠️ Skipping missing folder: {folder}")
        continue

    dataset = datasets.ImageFolder(folder, transform=test_transform)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    preds, labels, paths = [], [], []

    with torch.no_grad():
        for images, lbls in tqdm(dataloader, desc=f"Test_{test_idx}", leave=False):
            images = images.to(DEVICE)
            outputs = model(images)
            logits = outputs.logits
            predictions = torch.argmax(logits, dim=-1)

            preds.extend(predictions.cpu().numpy())
            labels.extend(lbls.numpy())
            paths.extend([path for path, _ in dataloader.dataset.samples])

    # Compute accuracy if labels are valid
    if len(labels) > 0:
        correct = sum(p == t for p, t in zip(preds, labels))
        acc = correct / len(labels)
        print(f"✅ Accuracy on {folder}: {acc:.4f}")
    else:
        acc = None

✅ Accuracy on ../snapshots/Test_0: 0.7315


✅ Accuracy on ../snapshots/Test_1: 0.7355


✅ Accuracy on ../snapshots/Test_2: 0.7153


✅ Accuracy on ../snapshots/Test_3: 0.6976


✅ Accuracy on ../snapshots/Test_4: 0.6038


✅ Accuracy on ../snapshots/Test_5: 0.7665


✅ Accuracy on ../snapshots/Test_6: 0.7355


✅ Accuracy on ../snapshots/Test_7: 0.6667


✅ Accuracy on ../snapshots/Test_8: 0.7355


✅ Accuracy on ../snapshots/Test_9: 0.6667
